# 02 — Climatological Features

| Feature | Source | Resolution | Method |
|---|---|---|---|
| Bio1: annual mean temp | CHELSAch (1981–2010) | ~1 km → 30 m | Temporal mean + bilinear regrid |
| Bio4: temp seasonality | CHELSAch (1981–2010) | ~1 km → 30 m | Temporal std + bilinear regrid |
| Bio12: annual precip sum | CHELSAch (1981–2010) | ~1 km → 30 m | Temporal sum + bilinear regrid |
| Bio15: precip seasonality | CHELSAch (1981–2010) | ~1 km → 30 m | CV (std/mean) + bilinear regrid |
| Bio18: warmest quarter precip | CHELSAch (1981–2010) | ~1 km → 30 m | Warmest 3 months sum + bilinear regrid |

Requires raw CHELSA NetCDF files in S3 (downloaded in the first section below).
Requires `full_dem` as the 30 m master grid.

### Download raw CHELSA data to S3

Stream raw CHELSA NetCDF files from source URLs directly into the S3 data lake.
Skip this section if the files are already in S3.

In [ ]:
import config
from s3_utils import stream_to_s3

stream_to_s3("data/envidatS3paths.txt", config.S3_BUCKET)

### Process CHELSA bioclimatic variables

In [ ]:
import config
from geo_utils import load_dem

full_dem = load_dem()


In [ ]:
import s3fs
from geo_utils import process_chelsa_variable, process_bio15, process_bio18

s3 = s3fs.S3FileSystem(anon=False)
base = config.S3_RAW + "/chelsa"
out = config.CHELSA_BASE


In [ ]:
# Bio1: annual mean temperature
process_chelsa_variable(
    full_dem, s3, base, f"{out}/bio1_30m.zarr",
    var_prefix="tas", aggregation="mean",
    dataset_name="bio1_annual_mean_temp",
)

# Bio4: temperature seasonality (std of monthly temps)
process_chelsa_variable(
    full_dem, s3, base, f"{out}/bio4_30m.zarr",
    var_prefix="tas", aggregation="std",
    dataset_name="bio4_temp_seasonality",
)

# Bio12: annual precipitation sum
process_chelsa_variable(
    full_dem, s3, base, f"{out}/bio12_30m.zarr",
    var_prefix="pr", aggregation="sum",
    dataset_name="bio12_annual_precipitation_sum",
)

# Bio15: precipitation seasonality (CV)
process_bio15(full_dem, s3, base, f"{out}/bio15_30m.zarr")

# Bio18: precipitation of warmest quarter
process_bio18(full_dem, s3, base, f"{out}/bio18_30m.zarr")

#### Visualize CHELSA results

In [ ]:
from plot_utils import plot_bio_from_zarr

plot_bio_from_zarr(
    f"{out}/bio18_30m.zarr",
    "bio18_warmest_quarter_precip",
    "Bio18: Precipitation of the Warmest Quarter",
    cmap="Blues",
)

plot_bio_from_zarr(
    f"{out}/bio1_30m.zarr",
    "bio1_annual_mean_temp",
    "Bio1: Annual Mean Temperature (30m)",
    cmap="RdYlBu_r", unit="K",
)